# Writting a Workflow Definition

In [1]:
import sys
import asyncio

### Business Logic

In [2]:
class GreetSomeone:
    async def run(self, name: str) -> str:
        return f"Hello {name}!"

In [3]:
name = "Algum Nome"
greeter = GreetSomeone()
greeting = await greeter.run(name)
print(greeting)

Hello Algum Nome!


## Workflow Definition

In [4]:
from temporalio import workflow


@workflow.defn
class GreetSomeone:
    @workflow.run
    async def run(self, name: str) -> str:
        return f"Hello {name}!"


# Criando o worker

In [5]:
import asyncio
import os

from temporalio.client import Client
from temporalio.worker import Worker, UnsandboxedWorkflowRunner
from temporalio import workflow

address = os.environ.get("TEMPORAL_ADDRESS", "localhost:7233")
client = await Client.connect(address, namespace="default")

worker = Worker(
    client,
    task_queue="greeting-tasks",
    workflows=[GreetSomeone],
    workflow_runner=UnsandboxedWorkflowRunner(),
)
worker_task = asyncio.ensure_future(worker.run())

print("Worker rodando em background...")

Worker rodando em background...


# Rodando o Workflow

rode 

```bash
make cli
```

e dentro, rode:

```bash
temporal workflow start --type GreetSomeone --task-queue greeting-tasks --workflow-id my-first-workflow --input '"Olá"'
```

O resultado deve ser algo como:

```
Running execution:
  WorkflowId  my-first-workflow
  RunId       019db845-4137-76bc-9f24-8044b7f87e63
  Type        GreetSomeone
  Namespace   default
  TaskQueue   greeting-tasks
```

Para recuperar o resultado, use o comando `temporal workflow show`:

```
$ temporal workflow show --workflow-id my-first-workflow
Progress:
  ID           Time                     Type           
    1  2026-04-22T21:41:20Z  WorkflowExecutionStarted  
    2  2026-04-22T21:41:20Z  WorkflowTaskScheduled     
    3  2026-04-22T21:41:20Z  WorkflowTaskStarted       
    4  2026-04-22T21:41:20Z  WorkflowTaskCompleted     
    5  2026-04-22T21:41:20Z  WorkflowExecutionCompleted

Results:
  Status          COMPLETED
  Result          "Hello Mason!"
  ResultEncoding  json/plain
```

# Rodar o Workflow via código

In [11]:
handle = await client.start_workflow(
    GreetSomeone.run,
    "Um nome bonito",
    id="greeting-workflow",
    task_queue="greeting-tasks",
)

print(f"Started workflow. Workflow ID: {handle.id}, RunID {handle.result_run_id}")

Started workflow. Workflow ID: greeting-workflow, RunID 019db86a-a2d4-7250-a9c2-41b09b446a2a


In [13]:
await handle.result()

'Hello Um nome bonito!'

Caso você queira bloquear o programa no momento da invocação do Workflow, pode condensar o código e usar o método `execute_workflow`, tornando o Workflow síncrono:

In [6]:
result = await client.execute_workflow(
    GreetSomeone.run,
    "Um nome bonito",
    id="greeting-workflow",
    task_queue="greeting-tasks",
)

print(f"Result: {result}")

Result: Hello Um nome bonito!


# Parando o Worker

In [15]:
worker_task.cancel()

True